In [1]:
## Problem Statement

# A leading biomedical research institute, NovaGen Research Labs, is conducting large-scale population health studies to better understand how underlying health conditions influence disease risk 
# and long-term health outcomes.
# Every year, the institute recruits thousands of volunteers who undergo:

# medical examinations
# lifestyle assessments
# clinical tests

# However, the researchers currently lack a reliable way to consistently distinguish between 
# individuals with generally healthy profiles and those who may be at higher health risk, 
# limiting the effectiveness of participant selection and stratified analysis in their studies.

# To address this challenge, the institute has compiled a dataset consisting of health records 
# from 9,800 individuals, collected over multiple observational studies.

# Each record represents a unique participant to avoid sampling bias and ensure independent observations.

# The dataset includes a combination of numerical and categorical health indicators, such as:

# physiological measurements
# lifestyle factors
# medical history attributes

# This provides a comprehensive view of each individual’s health status.

# You are engaged as a Data Scientist to develop a predictive model that classifies individuals 
# as “healthy” or “unhealthy” based on the available health data.

# This classification will support key research decisions, including:

# selecting eligible participants for clinical trials
# selecting participants for longitudinal studies
# Stratifying populations for risk-based analysis and outcome comparison

In [2]:
import pandas as pd
import numpy as np

In [3]:
data=pd.read_csv("novagen_dataset.csv")
data.head()

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,False,True,True,False,False
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,False,False,True,False,False
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,False,True,False,True,False


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

In [15]:
# data pre processing
categorical_cols=["Diet_Type__Vegan","Diet_Type__Vegetarian","Blood_Group_AB","Blood_Group_B","Blood_Group_O"]
numerical_cols=data.columns.difference(categorical_cols)
numerical_cols

Index(['Age', 'Alcohol', 'Allergies', 'BMI', 'Blood_Pressure', 'Cholesterol',
       'Diet', 'Exercise_Hours', 'Glucose_Level', 'Heart_Rate',
       'MedicalHistory', 'MentalHealth', 'PhysicalActivity', 'Sleep_Hours',
       'Smoking', 'Stress_Level', 'Target', 'Water_Intake'],
      dtype='object')

In [21]:
# do encoding for categorical_cols
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop="first", sparse_output=False)

encoded = ohe.fit_transform(data[categorical_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(categorical_cols)
)
encoded_df

,Diet_Type__Vegan_True,Diet_Type__Vegetarian_True,Blood_Group_AB_True,Blood_Group_B_True,Blood_Group_O_True
0,0.0,1.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...
9544,1.0,0.0,1.0,0.0,0.0
9545,0.0,1.0,0.0,1.0,0.0
9546,1.0,0.0,1.0,0.0,0.0
9547,1.0,0.0,1.0,0.0,0.0


In [22]:
data = pd.concat(
    [data.drop(columns=categorical_cols), encoded_df],
    axis=1
)
# encoded data is ready

In [25]:
data["Target"]

0       1
1       1
2       0
3       0
4       0
       ..
9544    0
9545    0
9546    0
9547    0
9548    0
Name: Target, Length: 9549, dtype: int64

In [26]:
# do scaling
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
sc=StandardScaler()


In [28]:
X=data.drop(columns=["Target"])
y=data["Target"]

Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.3,random_state=42)

In [29]:
Xtrain=sc.fit_transform(Xtrain)
Xtest=sc.transform(Xtest)

In [32]:
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score,classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

lr=LogisticRegression()
dt=DecisionTreeClassifier()
svc=SVC()

meta_model=LogisticRegression()

stacking=StackingClassifier(
    estimators=[
        ("lr",lr),
        ("dt",dt),
        ("svc",svc)
    ],
    cv=5,
    final_estimator=meta_model
)
stacking.fit(Xtrain,ytrain)
ypred1=stacking.predict(Xtest)
print("Stacking accuracy: ",accuracy_score(ypred1,ytest))

Stacking accuracy:  0.9445026178010472


In [33]:
from sklearn.ensemble import VotingClassifier
voting=VotingClassifier(
    estimators=[
        ("lr",lr),
        ("dt",dt),
        ("svc",svc)
    ]
)
voting.fit(Xtrain,ytrain)
ypred2=voting.predict(Xtest)
print("voting accuracy: ",accuracy_score(ypred2,ytest))

voting accuracy:  0.9232111692844677
